# Workshop Handoff: Triaje y Escalado Clínico (Hospital)

## 🎯 ¿Qué es el Patrón Handoff?

El patrón **Handoff** es una estrategia de orquestación donde **un agente evalúa la situación y decide dinámicamente a qué otro agente delegar la acción**.

A diferencia de Magentic (que itera y replantea), Handoff es una **decisión singular y clara**: un agente decide, el siguiente ejecuta.


## 🏥 Escenario del ejercicio (muy concreto)

Estás en **Urgencias** y recibes una solicitud de acción rápida:
> “Trasladar a radiología a un paciente con sospecha de sepsis para TAC, pero el traslado sería ~20 min **sin monitorización**”.

Lo que queremos **demostrar con Handoff** es esto:
1. Un **Coordinador** clasifica el **riesgo** (bajo/medio/alto) con una justificación breve.
2. Si el riesgo es **bajo/medio**, delega al **Equipo Asistencial** para proponer *qué haría y con qué condiciones* (medidas previas, vigilancia, escalado).
3. Si el riesgo es **alto**, no se ejecuta la acción: se delega directamente a **Registro Clínico** para dejar constancia del “no procede” con el motivo.
4. En cualquier caso, al final se registra el resultado en **Registro Clínico**.

### ✅ Qué deberías ver en la salida
- Una decisión explícita de `CoordinadorUrgencias`: `RIESGO: ...` + `JUSTIFICACIÓN: ...`
- Si se ejecuta: una respuesta breve de `EquipoAsistencial` con acciones/condiciones
- Siempre: una línea de `RegistroClinico` tipo `[ACCION]: [RESULTADO]`

---

## 1) Configuración del entorno
Verifica que tus variables de entorno están disponibles (Azure OpenAI).

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

base_url = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model_id = os.getenv("AZURE_OPENAI_DEPLOYMENT")

print("AZURE_OPENAI_ENDPOINT:", "OK" if base_url else "MISSING")
print("AZURE_OPENAI_API_KEY:", "OK" if api_key else "MISSING")
print("AZURE_OPENAI_DEPLOYMENT:", model_id if model_id else "MISSING")

AZURE_OPENAI_ENDPOINT: OK
AZURE_OPENAI_API_KEY: OK
AZURE_OPENAI_DEPLOYMENT: gpt-5.4


## 2) Concepto clave: Handoff

**Handoff** = el sistema decide qué agente actúa a continuación según el contexto.

En este taller:
- **Coordinador** decide el riesgo.
- **Ejecutor** ejecuta si procede.
- **Bitácora** registra el resultado.

El flujo se define con `HandoffBuilder` (MAF) y se ejecuta con `run_stream()`.

In [2]:
from agent_framework import ChatAgent, HandoffBuilder, WorkflowOutputEvent, AgentRunUpdateEvent
from agent_framework.openai import OpenAIChatClient

## 3) Ejercicio 1: Crear agentes

Crea los 3 roles básicos del flujo.

In [3]:
import inspect


def _make_chat_client():
    # Reducimos variabilidad si el SDK lo permite (mejora que siempre haga handoff).
    try:
        params = inspect.signature(OpenAIChatClient).parameters
    except Exception:
        params = {}

    kwargs = {
        "base_url": base_url,
        "api_key": api_key,
        "model_id": model_id,
    }
    if "temperature" in params:
        kwargs["temperature"] = 0
    return OpenAIChatClient(**kwargs)


coordinador = ChatAgent(
    chat_client=_make_chat_client(),
    name="CoordinadorUrgencias",
    instructions="""Eres el Coordinador de Urgencias (triaje y escalado).

OBJETIVO: decidir el riesgo y DELEGAR SIEMPRE a otro agente vía herramienta.

1) Primero, escribe EXACTAMENTE estas 2 líneas (y nada más):
RIESGO: [bajo/medio/alto]
JUSTIFICACIÓN: [1-2 oraciones]

2) Inmediatamente después, ejecuta EXACTAMENTE UNA herramienta de handoff:
- Si RIESGO es bajo o medio: usa `handoff_to_EquipoAsistencial`
- Si RIESGO es alto: usa `handoff_to_RegistroClinico`

REGLAS DURAS:
- No hagas preguntas al usuario.
- No pidas más información.
- No termines tu turno sin ejecutar una herramienta de handoff.
- No escribas el nombre de la herramienta como texto; ejecútala como tool-call.

Responde en español."""
 )

ejecutor = ChatAgent(
    chat_client=_make_chat_client(),
    name="EquipoAsistencial",
    instructions="""Eres el Equipo Asistencial. Solo actúas si el Coordinador delega en ti.

1) Escribe un plan breve y operativo (3-6 bullets cortos):
- medidas previas
- monitorización mínima
- criterio de no-traslado/escalado

2) Inmediatamente después, ejecuta la herramienta `handoff_to_RegistroClinico`.

REGLAS DURAS:
- No hagas preguntas al usuario.
- No pidas más información.
- No termines tu turno sin ejecutar `handoff_to_RegistroClinico`.
- No escribas el nombre de la herramienta como texto; ejecútala como tool-call.

Responde en español."""
 )

bitacora = ChatAgent(
    chat_client=_make_chat_client(),
    name="RegistroClinico",
    instructions="""Eres el Registro Clínico.
Responde con: [ACCION]: [RESULTADO] en 1 oración. Español."""
 )

print("Agentes listos:", coordinador.name, ejecutor.name, bitacora.name)

Agentes listos: CoordinadorUrgencias EquipoAsistencial RegistroClinico


## 4) Construir el workflow con HandoffBuilder

El flujo que montamos es deliberadamente simple y **trazable**:
- El **Coordinador** evalúa el riesgo.
- Si el riesgo es **bajo/medio**, delega al **Ejecutor**.
- Si el riesgo es **alto**, delega directamente a **Bitácora** (registro/rechazo).
- Tras ejecutar, siempre se registra en Bitácora.

```python
workflow = (
    HandoffBuilder(participants=[coordinador, ejecutor, bitacora])
    .set_coordinator(coordinador)
    .add_handoff(coordinador, [ejecutor, bitacora])
    .add_handoff(ejecutor, bitacora)
    .build()
 )
```

Mapa mental: `Coordinador → (Ejecutor | Bitácora)` y `Ejecutor → Bitácora`.

In [4]:
builder = (
    HandoffBuilder(participants=[coordinador, ejecutor, bitacora])
    .set_coordinator(coordinador)
    .add_handoff(coordinador, [ejecutor, bitacora])
    .add_handoff(ejecutor, [bitacora])
 )

# Evita que el workflow pida input al usuario entre agentes (si la API lo soporta)
_with_autonomous_mode = getattr(builder, "with_autonomous_mode", None)
if callable(_with_autonomous_mode):
    _with_autonomous_mode()

workflow = builder.build()

print("Workflow creado con HandoffBuilder (autonomous mode si disponible)")

Workflow creado con HandoffBuilder (autonomous mode si disponible)


## 5) Ejecutar el escenario

Aquí lanzamos el workflow con un caso **intencionalmente ambiguo** (tensión entre diagnóstico rápido vs seguridad).

**Objetivo del ejercicio**
- Ver cómo el Coordinador decide el riesgo y el workflow “se rutea” (handoff) hacia el agente correcto.
- Validar que el flujo siempre termina con un registro trazable (bitácora clínica).

**Qué tienes que mirar**
- En los eventos: cuándo ocurre el `HandoffEvent` (a quién se delega).
- En el resultado: qué contestó cada rol (`CoordinadorUrgencias`, `EquipoAsistencial`, `RegistroClinico`).

In [5]:
# Helpers extraídos de `ejecutar_un_escenario()` para una demo más explicable
from __future__ import annotations

import re
from typing import Any


def _recortar(texto: str, max_chars: int) -> str:
    texto = (texto or "").strip(" \t")
    if len(texto) <= max_chars:
        return texto
    return texto[: max_chars - 1].rstrip() + "…"


def _limpiar_preservando_saltos(texto: str) -> str:
    texto = (texto or "").replace("\r\n", "\n").replace("\r", "\n")
    lineas: list[str] = []
    for ln in texto.split("\n"):
        ln = " ".join(ln.split())
        if ln:
            lineas.append(ln)
    return "\n".join(lineas).strip()


def _extraer_texto_de_obj(obj: Any) -> str | None:
    if obj is None:
        return None

    for attr in ("output", "text", "content", "message", "delta"):
        if hasattr(obj, attr):
            valor = getattr(obj, attr)
            if valor is None:
                continue
            if isinstance(valor, list):
                txt = "".join(str(x) for x in valor if x is not None)
                return txt if txt.strip() else None
            if isinstance(valor, dict):
                if valor.get("text"):
                    return str(valor["text"])
                if valor.get("content"):
                    return str(valor["content"])
            txt = str(valor)
            if txt and txt != "None":
                return txt

    if hasattr(obj, "messages"):
        msgs = getattr(obj, "messages", None)
        if isinstance(msgs, list) and msgs:
            # Nos interesa el último mensaje útil
            for m in reversed(msgs):
                t = _extraer_texto_de_obj(m)
                if t and t.strip():
                    return t

    if isinstance(obj, str) and obj.strip():
        return obj

    return None


def _buscar_texto_profundo(obj: Any, *, max_depth: int = 5, max_items: int = 60) -> str | None:
    """Heurística robusta para extraer texto de estructuras desconocidas."""
    vistos: set[int] = set()

    def _inner(x: Any, depth: int, budget: list[int]) -> str | None:
        if x is None or depth < 0 or budget[0] <= 0:
            return None
        xid = id(x)
        if xid in vistos:
            return None
        vistos.add(xid)
        budget[0] -= 1

        if isinstance(x, str):
            return x if x.strip() else None
        if isinstance(x, dict):
            # Prioriza claves típicas
            for k in ("text", "content", "message", "output", "delta"):
                if k in x and isinstance(x[k], str) and x[k].strip():
                    return x[k]
            for v in x.values():
                t = _inner(v, depth - 1, budget)
                if t and t.strip():
                    return t
            return None
        if isinstance(x, (list, tuple)):
            for v in x:
                t = _inner(v, depth - 1, budget)
                if t and t.strip():
                    return t
            return None

        # Objetos: intenta attrs frecuentes
        for attr in ("text", "content", "message", "output", "delta", "result", "final", "value"):
            if hasattr(x, attr):
                t = _inner(getattr(x, attr), depth - 1, budget)
                if t and t.strip():
                    return t
        if hasattr(x, "messages"):
            t = _inner(getattr(x, "messages"), depth - 1, budget)
            if t and t.strip():
                return t
        return None

    return _inner(obj, max_depth, [max_items])


def _extraer_actor(event: Any) -> str | None:
    # Preferimos el nombre del agente (más estable) frente a executor_id (que puede ser interno: handoff-..., input-...)
    for obj in (getattr(event, "data", None), event):
        if obj is None:
            continue
        actor = (
            getattr(obj, "agent_name", None)
            or getattr(obj, "author_name", None)
            or getattr(obj, "name", None)
            or getattr(obj, "executor_id", None)
        )
        if actor:
            return str(actor)
    return None


def _extraer_texto_evento(event: Any) -> str | None:
    data = getattr(event, "data", None)
    candidates = [
        data,
        getattr(data, "agent_run_response", None),
        getattr(data, "message", None),
        getattr(data, "update", None),
        getattr(data, "response", None),
        getattr(data, "result", None),
        getattr(event, "message", None),
        getattr(event, "result", None),
    ]
    for c in candidates:
        texto = _extraer_texto_de_obj(c)
        if texto and texto.strip():
            return texto
    return _buscar_texto_profundo(data)


_RE_RIESGO = re.compile(r"\bRIESGO\s*:\s*(bajo|medio|alto)\b", re.IGNORECASE)


def _extraer_riesgo(texto: str | None) -> str | None:
    if not texto:
        return None
    m = _RE_RIESGO.search(texto)
    if not m:
        return None
    return m.group(1).lower()


def _imprimir_bloque(
    titulo: str,
    texto: str,
    *,
    max_lineas_por_bloque: int,
    max_chars_por_bloque: int,
) -> None:
    texto = _recortar(texto, max_chars_por_bloque)
    if not texto.strip():
        return
    print(f"     ↳ {titulo}:")
    texto = texto.replace("\r", "")
    ancho = 120
    lineas = 0
    i = 0
    while i < len(texto) and lineas < max_lineas_por_bloque:
        frag = texto[i : i + ancho]
        print(f"        {frag}")
        i += ancho
        lineas += 1
    if i < len(texto):
        print("        …")


def _deberia_flush(buffer: str, delta: str, *, umbral_flush_chars: int) -> bool:
    if "\n" in delta:
        return True
    if len(buffer) >= umbral_flush_chars:
        return True
    if delta and delta[-1] in (".", "!", "?", ":", ";"):
        return True
    return False


def _append_delta(
    buffers: dict[str, str],
    acumulado: dict[str, str],
    actor: str,
    delta: str,
    *,
    max_chars_por_agente_final: int,
) -> None:
    if not delta:
        return
    buffers[actor] = buffers.get(actor, "") + delta
    acumulado[actor] = acumulado.get(actor, "") + delta
    if len(acumulado[actor]) > max_chars_por_agente_final:
        acumulado[actor] = acumulado[actor][-max_chars_por_agente_final:]


def _flush(
    buffers: dict[str, str],
    actor: str,
    *,
    fuerza: bool = False,
    min_chars: int = 20,
) -> str:
    texto = buffers.get(actor, "")
    if not texto:
        return ""
    if not fuerza and len(texto.strip()) < min_chars:
        return ""
    texto = _limpiar_preservando_saltos(texto)
    buffers[actor] = ""
    return texto


async def run_stream_handoff_con_traza(
    workflow,
    entrada: str,
    *,
    buffers: dict[str, str],
    acumulado: dict[str, str],
    event_map: dict[str, str],
    max_lineas_por_bloque: int = 10,
    max_chars_por_bloque: int = 1400,
    max_chars_por_agente_final: int = 4000,
    umbral_flush_chars: int = 240,
):
    """
    Ejecuta `workflow.run_stream(entrada)` e imprime una traza legible.

    Importante para la demo:
    - No captura excepciones (el `try/except` vive en la celda principal).
    - No hace el "flush final" (vive en la celda principal).
    """
    output_evt = None
    n_evento = 0

    riesgo_coordinador: str | None = None
    ya_verificado_ruteo = False

    async for event in workflow.run_stream(entrada):
        n_evento += 1
        event_type = type(event).__name__
        actor = _extraer_actor(event) or "SISTEMA"

        # Para updates de streaming: NO imprimimos cabecera por delta; solo cuando hay bloque
        if isinstance(event, AgentRunUpdateEvent):
            delta = _extraer_texto_evento(event) or ""
            if delta:
                _append_delta(
                    buffers,
                    acumulado,
                    actor,
                    delta,
                    max_chars_por_agente_final=max_chars_por_agente_final,
                )
                if _deberia_flush(buffers.get(actor, ""), delta, umbral_flush_chars=umbral_flush_chars):
                    bloque = _flush(buffers, actor)
                    if bloque:
                        print(f"[{n_evento:2d}] {actor:18s} | 🤖 Texto (stream)")
                        _imprimir_bloque(
                            "texto",
                            bloque,
                            max_lineas_por_bloque=max_lineas_por_bloque,
                            max_chars_por_bloque=max_chars_por_bloque,
                        )
            continue

        desc = event_map.get(event_type, f"⚡ {event_type}")
        print(f"[{n_evento:2d}] {actor:18s} | {desc}")

        if event_type == "HandoffEvent":
            origen = (
                getattr(event, "from_executor_id", None)
                or getattr(event, "from_agent", None)
                or getattr(event, "source", None)
            )
            destino = (
                getattr(event, "to_executor_id", None)
                or getattr(event, "to_agent", None)
                or getattr(event, "target", None)
            )
            if origen or destino:
                print(f"     ↳ Ruta: {origen or '?'} → {destino or '?'}")

            # Verificación explícita del ruteo del Coordinador (si ya conocemos el riesgo)
            if (not ya_verificado_ruteo) and riesgo_coordinador and origen and "Coordinador" in str(origen):
                esperado = "EquipoAsistencial" if riesgo_coordinador in ("bajo", "medio") else "RegistroClinico"
                real = str(destino or "?")
                ok = esperado in real
                print(f"     ↳ Verificación ruteo: RIESGO={riesgo_coordinador} | esperado→ {esperado} | real→ {real}")
                if not ok:
                    print("     ↳ [AVISO] El ruteo NO coincide con la regla. Esto suele pasar si el LLM no ejecutó el tool correcto o eligió otro destinatario.")
                ya_verificado_ruteo = True

        if event_type == "ExecutorCompletedEvent":
            pendiente = _flush(buffers, actor, fuerza=True)
            if pendiente:
                _imprimir_bloque(
                    "texto",
                    pendiente,
                    max_lineas_por_bloque=max_lineas_por_bloque,
                    max_chars_por_bloque=max_chars_por_bloque,
                )

            full = _extraer_texto_evento(event)
            if not full:
                full = _buscar_texto_profundo(getattr(event, "data", None))
            if full:
                acumulado[actor] = full
            else:
                acumulado[actor] = _limpiar_preservando_saltos(acumulado.get(actor, ""))

            if actor == "CoordinadorUrgencias":
                riesgo_coordinador = _extraer_riesgo(acumulado.get(actor)) or riesgo_coordinador

        if isinstance(event, WorkflowOutputEvent):
            output_evt = event
            break

    return output_evt


def imprimir_resultado_final(
    output_evt,
    acumulado: dict[str, str],
    roles_ordenados: list[str],
) -> None:
    if output_evt and getattr(output_evt, "data", None):
        mensajes = list(output_evt.data or [])
        por_autor: dict[str, str] = {}
        for msg in mensajes:
            autor = getattr(msg, "author_name", None) or "Asistente"
            contenido = msg.text if hasattr(msg, "text") else str(getattr(msg, "content", ""))
            por_autor[autor] = _limpiar_preservando_saltos(contenido)

        autores_orden = [str(a) for a in roles_ordenados if a in por_autor] + [
            str(a) for a in por_autor.keys() if a not in roles_ordenados
        ]
        print("Autores detectados:", ", ".join(autores_orden) if autores_orden else "(ninguno)")
        faltan = [r for r in roles_ordenados if r not in por_autor]
        if faltan:
            print("[!] Falta output de:", ", ".join(faltan))
            print("    (Suele indicar que no se hizo handoff o que el workflow se detuvo esperando input.)\n")

        idx = 0
        for rol in roles_ordenados:
            if rol in por_autor and por_autor[rol].strip():
                idx += 1
                print(f"[{idx}] {rol}:")
                print(f"    {_recortar(por_autor[rol], 2600)}\n")

        for autor, contenido in por_autor.items():
            if autor in roles_ordenados:
                continue
            if contenido.strip():
                idx += 1
                print(f"[{idx}] {autor}:")
                print(f"    {_recortar(contenido, 2600)}\n")

    elif acumulado:
        print("Autores detectados (fallback):", ", ".join(str(a) for a in list(acumulado.keys())[:10]))
        idx = 0
        for rol in roles_ordenados:
            if rol in acumulado:
                texto = _limpiar_preservando_saltos(acumulado[rol])
                if texto.strip():
                    idx += 1
                    print(f"[{idx}] {rol}:")
                    print(f"    {_recortar(texto, 2600)}\n")

        for actor, texto in acumulado.items():
            if actor in roles_ordenados:
                continue
            texto = _limpiar_preservando_saltos(texto)
            if texto.strip():
                idx += 1
                print(f"[{idx}] {actor}:")
                print(f"    {_recortar(texto, 2600)}\n")
    else:
        print("[!] No hay respuestas de agentes\n")

In [6]:
#- Sospecha de sepsis en paciente adulto
#- Hemodinámica límite (hipotensión)

async def ejecutar_un_escenario():
    entrada = """
CASO (Urgencias):

- Se solicita TAC en radiología
- Traslado estimado: 20 minutos
- Restricción: el traslado sería SIN monitorización continua

Pregunta: ¿es seguro proceder ahora? Si procede, ¿bajo qué condiciones/medidas previas?
"""

    print("Entrada:\n", entrada)
    print("-" * 80)
    print("TRAZA (con detalle):\n")

    event_map = {
        "WorkflowStartedEvent": "🚀 Workflow iniciado",
        "WorkflowStatusEvent": "📊 Estado del workflow",
        "SuperStepStartedEvent": "🧩 SuperStep iniciado",
        "SuperStepCompletedEvent": "🧩 SuperStep completado",
        "ExecutorInvokedEvent": "⚙️  Executor invocado",
        "ExecutorCompletedEvent": "✅ Executor completado",
        "RequestInfoEvent": "❓ Solicitando entrada",
        "HandoffEvent": "🔁 Transición (handoff)",
        "AgentRunUpdateEvent": "🤖 Update de agente",
        "WorkflowOutputEvent": "🏁 Workflow completado",
    }

    # Control de detalle (más informativo SIN imprimir deltas en micro-trozos)
    MAX_LINEAS_POR_BLOQUE = 10
    MAX_CHARS_POR_BLOQUE = 1400
    MAX_CHARS_POR_AGENTE_FINAL = 4000
    UMBRAL_FLUSH_CHARS = 240  # cuando el buffer crece, volcamos un bloque

    # Orden preferido en el resumen final (roles del ejercicio)
    roles_ordenados = [str(coordinador.name), str(ejecutor.name), str(bitacora.name)]

    # Estos DOS elementos se quedan en la celda principal por claridad en la demo:
    # - try/except
    # - flush final
    buffers: dict[str, str] = {}
    acumulado: dict[str, str] = {}
    output_evt = None

    try:
        output_evt = await run_stream_handoff_con_traza(
            workflow,
            entrada,
            buffers=buffers,
            acumulado=acumulado,
            event_map=event_map,
            max_lineas_por_bloque=MAX_LINEAS_POR_BLOQUE,
            max_chars_por_bloque=MAX_CHARS_POR_BLOQUE,
            max_chars_por_agente_final=MAX_CHARS_POR_AGENTE_FINAL,
            umbral_flush_chars=UMBRAL_FLUSH_CHARS,
        )
    except Exception as e:
        print(f"[ERROR] {str(e)}")

    # Flush final de cualquier buffer pendiente (sin micro-deltas)
    for actor in list(buffers.keys()):
        bloque = _flush(buffers, actor, fuerza=True)
        if bloque:
            print(f"[--] {actor:18s} | 🤖 Texto (stream)")
            _imprimir_bloque(
                "texto",
                bloque,
                max_lineas_por_bloque=MAX_LINEAS_POR_BLOQUE,
                max_chars_por_bloque=MAX_CHARS_POR_BLOQUE,
            )

    print("\n" + "-" * 80)
    print("\nRESULTADO FINAL (por rol):\n")
    imprimir_resultado_final(output_evt, acumulado, roles_ordenados)


await ejecutar_un_escenario()

Entrada:
 
CASO (Urgencias):

- Se solicita TAC en radiología
- Traslado estimado: 20 minutos
- Restricción: el traslado sería SIN monitorización continua

Pregunta: ¿es seguro proceder ahora? Si procede, ¿bajo qué condiciones/medidas previas?

--------------------------------------------------------------------------------
TRAZA (con detalle):

[ 1] SISTEMA            | 🚀 Workflow iniciado
[ 2] SISTEMA            | 📊 Estado del workflow
[ 3] input-conversation | ⚙️  Executor invocado
[ 4] input-conversation | ✅ Executor completado
[ 5] SISTEMA            | 🧩 SuperStep iniciado
[ 6] CoordinadorUrgencias | ⚙️  Executor invocado
[16] CoordinadorUrgencias | 🤖 Texto (stream)
     ↳ texto:
        RIESGO: altoJUSTIFICACIÓN:
[46] CoordinadorUrgencias | 🤖 Texto (stream)
     ↳ texto:
        El traslado a TAC sin monitorización continua en un paciente de urgencias implica riesgo significativo de deterioro no d
        etectado durante al menos20 minutos.
[70] CoordinadorUrgencias | 🤖 Texto (s

## 6) Cómo interpretar la salida

- **Eventos**: indican los pasos internos del workflow.
- **HandoffEvent**: transición entre agentes.
- **WorkflowOutputEvent**: salida final agregada.

> Si ves `RequestInfoEvent`, significa que el flujo espera más input del usuario.